In [3]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Анализ распределения классов дефектов в train.csv (Severstal)
Подсчет количества изображений и масок для каждого класса
"""

import pandas as pd
import numpy as np
from collections import defaultdict
from pathlib import Path

# ============================================
# 1. ЗАГРУЗКА ДАННЫХ
# ============================================

CSV_PATH = Path("data/severstal/train.csv")

print("="*80)
print("АНАЛИЗ РАСПРЕДЕЛЕНИЯ КЛАССОВ В TRAIN.CSV (Severstal)")
print("="*80)

# Загрузка CSV
df = pd.read_csv(CSV_PATH)
print(f"\n📁 Загружено строк: {len(df):,}")
print(f"📁 Уникальных изображений: {df['ImageId'].nunique():,}")

АНАЛИЗ РАСПРЕДЕЛЕНИЯ КЛАССОВ В TRAIN.CSV (Severstal)

📁 Загружено строк: 7,095
📁 Уникальных изображений: 6,666


In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Анализ распределения классов дефектов в train.csv (Severstal)
Подсчет количества изображений и масок для каждого класса
"""

import pandas as pd
import numpy as np
from collections import defaultdict
from pathlib import Path

# ============================================
# 1. ЗАГРУЗКА ДАННЫХ
# ============================================

CSV_PATH = Path("data/severstal/train.csv")

print("="*80)
print("АНАЛИЗ РАСПРЕДЕЛЕНИЯ КЛАССОВ В TRAIN.CSV (Severstal)")
print("="*80)

# Загрузка CSV
df = pd.read_csv(CSV_PATH)
print(f"\n📁 Загружено строк: {len(df):,}")
print(f"📁 Уникальных изображений: {df['ImageId'].nunique():,}")

# ============================================
# 2. ПОДСЧЕТ СТАТИСТИК ПО КЛАССАМ
# ============================================

# Словари для хранения статистики
images_per_class = defaultdict(set)  # уникальные изображения для каждого класса
masks_per_class = defaultdict(int)   # количество масок (EncodedPixels) для каждого класса
total_pixels_per_class = defaultdict(int)  # общее количество пикселей дефектов

# Классы в датасете Severstal (согласно документации)
CLASS_NAMES = {
    1: "Дефект 1",
    2: "Дефект 2", 
    3: "Дефект 3",
    4: "Дефект 4"
}

# Обработка каждой строки
for _, row in df.iterrows():
    image_id = row['ImageId']
    class_id = row['ClassId']
    encoded_pixels = row['EncodedPixels']
    
    # Добавляем изображение в множество для этого класса
    images_per_class[class_id].add(image_id)
    
    # Считаем маски (если EncodedPixels не пустой)
    if pd.notna(encoded_pixels) and str(encoded_pixels).strip():
        masks_per_class[class_id] += 1
        
        # Подсчет пикселей в маске (RLE формат: пары [пиксель, длина])
        pixels = str(encoded_pixels).split()
        if len(pixels) >= 2:
            for i in range(1, len(pixels), 2):
                total_pixels_per_class[class_id] += int(pixels[i])

# ============================================
# 3. ВЫВОД СТАТИСТИКИ
# ============================================

print(f"\n{'='*80}")
print("СТАТИСТИКА ПО КЛАССАМ")
print(f"{'='*80}")

total_images = df['ImageId'].nunique()
total_masks = sum(masks_per_class.values())

print(f"\n📊 ОБЩАЯ СТАТИСТИКА:")
print(f"  Всего изображений в датасете: {total_images:,}")
print(f"  Всего масок дефектов: {total_masks:,}")

print(f"\n📊 РАСПРЕДЕЛЕНИЕ ПО КЛАССАМ:\n")
print(f"{'Класс':<10} {'Название':<15} {'Изображений':<15} {'Масок':<15} {'% масок':<12} {'Пикселей':<15}")
print("-" * 85)

for class_id in sorted(CLASS_NAMES.keys()):
    name = CLASS_NAMES[class_id]
    img_count = len(images_per_class[class_id])
    mask_count = masks_per_class[class_id]
    mask_pct = (mask_count / total_masks * 100) if total_masks > 0 else 0
    pixel_count = total_pixels_per_class[class_id]
    
    print(f"{class_id:<10} {name:<15} {img_count:<15,} {mask_count:<15,} {mask_pct:<11.1f}% {pixel_count:<15,}")

# ============================================
# 4. АНАЛИЗ СОВМЕСТНОГО ВСТРЕЧАНИЯ ДЕФЕКТОВ
# ============================================

print(f"\n{'='*80}")
print("АНАЛИЗ СОВМЕСТНОГО ВСТРЕЧАНИЯ ДЕФЕКТОВ")
print(f"{'='*80}")

# Собираем все классы для каждого изображения
image_classes = defaultdict(set)
for _, row in df.iterrows():
    if pd.notna(row['EncodedPixels']):
        image_classes[row['ImageId']].add(row['ClassId'])

# Подсчет количества классов на изображение
classes_per_image = [len(classes) for classes in image_classes.values()]
images_with_multiple = sum(1 for c in classes_per_image if c > 1)
images_with_single = sum(1 for c in classes_per_image if c == 1)

print(f"\n  Изображений с 1 классом дефектов: {images_with_single:,} ({images_with_single/len(image_classes)*100:.1f}%)")
print(f"  Изображений с 2+ классами дефектов: {images_with_multiple:,} ({images_with_multiple/len(image_classes)*100:.1f}%)")

# Топ комбинаций классов
from itertools import combinations
combo_counts = defaultdict(int)
for classes in image_classes.values():
    if len(classes) >= 2:
        for combo in combinations(sorted(classes), 2):
            combo_counts[combo] += 1

if combo_counts:
    print(f"\n  Топ-5 пар дефектов, встречающихся вместе:")
    for (c1, c2), count in sorted(combo_counts.items(), key=lambda x: x[1], reverse=True)[:5]:
        name1 = CLASS_NAMES[c1]
        name2 = CLASS_NAMES[c2]
        print(f"    {name1} + {name2}: {count} изображений")

# ============================================
# 5. АНАЛИЗ ЧИСТЫХ ИЗОБРАЖЕНИЙ
# ============================================

print(f"\n{'='*80}")
print("АНАЛИЗ ЧИСТЫХ ИЗОБРАЖЕНИЙ")
print(f"{'='*80}")

# Изображения без дефектов (все ClassId, но с пустыми EncodedPixels)
all_image_ids = set(df['ImageId'].unique())
images_with_defects = set(image_classes.keys())
clean_images = all_image_ids - images_with_defects

print(f"\n  Чистых изображений (без дефектов): {len(clean_images):,}")
print(f"  Изображений с дефектами: {len(images_with_defects):,}")
print(f"  Соотношение дефектные/чистые: 1:{len(clean_images)/len(images_with_defects):.2f}")

# ============================================
# 6. СТАТИСТИКА ПО РАЗМЕРАМ ДЕФЕКТОВ
# ============================================

print(f"\n{'='*80}")
print("СТАТИСТИКА ПО РАЗМЕРАМ ДЕФЕКТОВ")
print(f"{'='*80}")

# Собираем размеры масок для каждого класса
mask_sizes_by_class = defaultdict(list)

for _, row in df.iterrows():
    class_id = row['ClassId']
    encoded_pixels = row['EncodedPixels']
    
    if pd.notna(encoded_pixels) and str(encoded_pixels).strip():
        pixels = str(encoded_pixels).split()
        mask_size = 0
        for i in range(1, len(pixels), 2):
            mask_size += int(pixels[i])
        mask_sizes_by_class[class_id].append(mask_size)

print(f"\n  Размеры дефектов (в пикселях):\n")
print(f"{'Класс':<10} {'Мин. размер':<15} {'Средний':<15} {'Медиана':<15} {'Макс. размер':<15}")
print("-" * 75)

for class_id in sorted(CLASS_NAMES.keys()):
    sizes = mask_sizes_by_class[class_id]
    if sizes:
        name = CLASS_NAMES[class_id]
        print(f"{class_id:<10} {min(sizes):<15,} {np.mean(sizes):<15,.0f} {np.median(sizes):<15,.0f} {max(sizes):<15,}")

# ============================================
# 7. ВЫВОДЫ И РЕКОМЕНДАЦИИ
# ============================================

print(f"\n{'='*80}")
print("ВЫВОДЫ")
print(f"{'='*80}")

# Находим самый частый и самый редкий класс
most_common = max(masks_per_class.items(), key=lambda x: x[1])
least_common = min(masks_per_class.items(), key=lambda x: x[1])

print(f"""
1. РАСПРЕДЕЛЕНИЕ КЛАССОВ:
   - Самый частый класс: {most_common[0]} ({CLASS_NAMES[most_common[0]]}) - {most_common[1]:,} масок ({most_common[1]/total_masks*100:.1f}%)
   - Самый редкий класс: {least_common[0]} ({CLASS_NAMES[least_common[0]]}) - {least_common[1]:,} масок ({least_common[1]/total_masks*100:.1f}%)
   - Дисбаланс: класс {most_common[0]} встречается в {most_common[1]/least_common[1]:.1f} раз чаще класса {least_common[0]}

2. СТРУКТУРА ДАННЫХ:
   - Всего изображений: {total_images:,}
   - С дефектами: {len(images_with_defects):,} ({len(images_with_defects)/total_images*100:.1f}%)
   - Без дефектов: {len(clean_images):,} ({len(clean_images)/total_images*100:.1f}%)

3. СЛОЖНОСТЬ РАЗМЕТКИ:
   - Изображений с одним классом: {images_with_single:,}
   - Изображений с несколькими классами: {images_with_multiple:,}

4. РЕКОМЕНДАЦИИ:
   - Для класса {least_common[0]} требуется аугментация/генерация синтетики
   - Учитывать возможность множественных дефектов на одном изображении
   - Использовать stratified sampling при разбиении на train/val/test
""")

print("="*80)
print("✅ Анализ завершен!")

АНАЛИЗ РАСПРЕДЕЛЕНИЯ КЛАССОВ В TRAIN.CSV (Severstal)

📁 Загружено строк: 7,095
📁 Уникальных изображений: 6,666

СТАТИСТИКА ПО КЛАССАМ

📊 ОБЩАЯ СТАТИСТИКА:
  Всего изображений в датасете: 6,666
  Всего масок дефектов: 7,095

📊 РАСПРЕДЕЛЕНИЕ ПО КЛАССАМ:

Класс      Название        Изображений     Масок           % масок      Пикселей       
-------------------------------------------------------------------------------------
1          Дефект 1        897             897             12.6       % 3,912,129      
2          Дефект 2        247             247             3.5        % 834,471        
3          Дефект 3        5,150           5,150           72.6       % 131,306,899    
4          Дефект 4        801             801             11.3       % 27,533,572     

АНАЛИЗ СОВМЕСТНОГО ВСТРЕЧАНИЯ ДЕФЕКТОВ

  Изображений с 1 классом дефектов: 6,239 (93.6%)
  Изображений с 2+ классами дефектов: 427 (6.4%)

  Топ-5 пар дефектов, встречающихся вместе:
    Дефект 3 + Дефект 4: 284 изображ